# A SAIA-backed OpenAI-compatible proxy (example)

This notebook builds a tiny **OpenAI-compatible HTTP proxy** in front of the
GWDG SAIA platform using `saia_python`. Any OpenAI-SDK app — the `openai`
Python client, LibreChat, Cursor, … — can then point at the proxy, talk to
SAIA, and get **clean, numbered citations** instead of GWDG ARCANA's verbose
`References:` dump.

> **This is an example, not part of `saia_python`'s core.** `saia_python` is a
> *client* library; a server (FastAPI) and its dependencies stay out of it.
> The proxy here is assembled from the library's building blocks:
> - `client.models.list_raw()` → serve `/v1/models` (the real SAIA list)
> - `client.arcana.chat(...)` / `client.chat.completions(...)` → forward chat
> - `saia_python.arcana_references.parse_arcana_references()` → clean citations
>
> See **ADR-0005** for why the GWDG reference parser lives in the library while
> serving/rendering stays with the consumer.

**Use case.** A clinician-facing tool speaks the OpenAI API. We want it to query
a SAIA ARCANA knowledge base and show a tidy source list — without teaching the
tool anything SAIA-specific. The proxy does the translation centrally, so every
client benefits.


## Architecture

```
OpenAI-SDK client ──/v1/*──▶   this proxy (FastAPI)   ──▶   GWDG SAIA
  (openai, LibreChat)          • GET  /v1/models  = list_raw()
                               • POST /v1/chat/completions
                                       = forward, then clean references
```

The forwarding is trivial; the value is the citation cleanup. Everything
product-specific (auth, prompt injection, filename→URL links, UI quirks) would
live in *your* adapter, not here.


In [ ]:
# --- Setup ---------------------------------------------------------------
# Core dependency: saia_python (its pure ARCANA reference parser).
import re
from saia_python.arcana_references import parse_arcana_references

# The proxy needs FastAPI (an *example* dep, NOT a saia_python dependency).
try:
    from fastapi import FastAPI, Request
    from fastapi.testclient import TestClient
    HAVE_FASTAPI = True
except ImportError:
    HAVE_FASTAPI = False
    print("fastapi not installed -> `pip install fastapi httpx` for sections 2-4")

# A realistic ARCANA-routed reply: a short answer + GWDG's verbose References
# dump (one `[RREFn] <file>.md (<distance>)` line per chunk, each trailed by
# the chunk body).
GWDG_SAMPLE = (
    "R-CHOP is the standard first-line therapy for DLBCL [RREF1]; interim "
    "PET-CT guides response assessment [RREF2].\n\n"
    "This research aid does not replace medical advice.\n"
    "-----------------------------------------\n"
    "References:\n\n"
    "[RREF1] onkopedia_dlbcl_5-1-1_immunochemotherapy__ID0EAB.md (0.312)\n"
    "## Immunochemotherapy\n(... a few hundred lines of chunk body ...)\n"
    "[RREF2] onkopedia_dlbcl_5-2_response-assessment__ID0ECD.md (0.357)\n"
    "(... another long chunk body ...)\n"
)
print(GWDG_SAMPLE)


## 1. Clean ARCANA references (offline — no network, no key)

`parse_arcana_references()` splits the assistant content into the answer prose
and a list of structured `ArcanaReference`s (`n`, `filename`, `distance`). We
render those into a compact, numbered **Sources** block — what a downstream
OpenAI client should actually see. Interpreting the filename (into a URL or a
nicer label) is your app's job; here we just list it.


In [ ]:
def clean_content(content: str) -> str:
    # Replace GWDG's verbose `References:` dump with a compact numbered list.
    parsed = parse_arcana_references(content)
    if not parsed.matched or not parsed.references:
        return content                       # no ARCANA refs -> pass through
    # prose = everything before the marker; drop GWDG's trailing rule line.
    prose = re.sub(r"\n-{3,}\s*$", "", parsed.prose).rstrip("\n")
    lines = ["Sources:"]
    for ref in parsed.references:
        score = "" if ref.distance is None else f" (score {ref.distance:.3f})"
        lines.append(f"{ref.n}. {ref.filename}{score}")
    return prose + "\n\n" + "\n".join(lines)


print(clean_content(GWDG_SAMPLE))


## 2. The proxy app

Two endpoints:

- `GET /v1/models` re-serves SAIA's model list verbatim, so a client configured
  with `models: { fetch: true }` (LibreChat) discovers models automatically.
- `POST /v1/chat/completions` forwards the request to SAIA, then runs the reply
  through `clean_content()`.

A thin `backend` seam lets the same app run against a **mock** SAIA (offline,
next) or a **real** `SAIAClient` (the "go live" section).


In [ ]:
def build_proxy(backend):
    # backend.models()    -> dict  (an OpenAI /v1/models envelope)
    # backend.chat(body)  -> dict  (an OpenAI chat-completion response)
    app = FastAPI(title="saia-openai-proxy (example)")

    @app.get("/v1/models")
    def list_models():
        return backend.models()

    @app.post("/v1/chat/completions")
    async def chat_completions(request: Request):
        body = await request.json()
        resp = backend.chat(body)
        for choice in resp.get("choices", []):
            msg = choice.get("message") or {}
            if isinstance(msg.get("content"), str):
                msg["content"] = clean_content(msg["content"])   # <- the cleanup
        return resp

    return app


print("build_proxy ready" if HAVE_FASTAPI else "skipped: fastapi not installed")


## 3. Run it in-process against a *mock* SAIA (fully offline)

No network, no API key. We stub the backend to return `GWDG_SAMPLE` and use
FastAPI's `TestClient` to call the proxy exactly as an OpenAI client would —
proving the end-to-end shape: a verbose GWDG reply goes in, a clean OpenAI
response comes out.


In [ ]:
class MockSAIA:
    # Stand-in for SAIAClient: returns canned OpenAI-shaped payloads.
    def models(self):
        return {"object": "list", "data": [
            {"id": "meta-llama-3.1-8b-instruct", "object": "model", "owned_by": "saia"},
        ]}

    def chat(self, body):
        return {
            "id": "chatcmpl-demo",
            "object": "chat.completion",
            "model": body.get("model", "meta-llama-3.1-8b-instruct"),
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": GWDG_SAMPLE},
                "finish_reason": "stop",
            }],
        }


if HAVE_FASTAPI:
    proxy = TestClient(build_proxy(MockSAIA()))

    print("=== GET /v1/models ===")
    print(proxy.get("/v1/models").json())

    print("\n=== POST /v1/chat/completions (what the OpenAI client receives) ===")
    r = proxy.post("/v1/chat/completions", json={
        "model": "meta-llama-3.1-8b-instruct",
        "messages": [{"role": "user", "content": "First-line therapy for DLBCL?"}],
    })
    print(r.json()["choices"][0]["message"]["content"])
else:
    print("Install fastapi to run this cell: pip install fastapi httpx")


## 4. Go live against real SAIA (optional)

To run against your real SAIA account, back the proxy with a `SAIAClient`, serve
it with uvicorn, and point the `openai` SDK at it. This needs a SAIA API key,
network access, and `pip install uvicorn openai`. It is **guarded** by
`RUN_LIVE` so "Run all" stays offline-safe.


In [ ]:
import os

RUN_LIVE = bool(os.environ.get("SAIA_API_KEY")) and os.environ.get("RUN_LIVE") == "1"


class SAIABackend:
    # Real backend on the native (synchronous) saia_python client.
    # Pass an arcana_id to route through an ARCANA knowledge base (RAG).
    def __init__(self, arcana_id=None):
        from saia_python import SAIAClient
        self._client = SAIAClient()          # key from env / .env / config.toml
        self._arcana_id = arcana_id

    def models(self):
        return self._client.models.list_raw()

    def chat(self, body):
        model = body.get("model", "meta-llama-3.1-8b-instruct")
        messages = body.get("messages", [])
        if self._arcana_id:
            return self._client.arcana.chat(
                model=model, messages=messages, arcana_id=self._arcana_id)
        return self._client.chat.completions(model=model, messages=messages)


if RUN_LIVE:
    import threading, time, uvicorn
    app = build_proxy(SAIABackend(arcana_id=os.environ.get("SAIA_ARCANA_ID")))
    server = uvicorn.Server(uvicorn.Config(
        app, host="127.0.0.1", port=8011, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    time.sleep(1.5)                          # let the server boot

    from openai import OpenAI
    oai = OpenAI(base_url="http://127.0.0.1:8011/v1", api_key="proxy-ignores-this")
    print("models:", [m.id for m in oai.models.list().data][:5])
    out = oai.chat.completions.create(
        model="meta-llama-3.1-8b-instruct",
        messages=[{"role": "user", "content": "First-line therapy for DLBCL?"}],
    )
    print(out.choices[0].message.content)
else:
    print("Set SAIA_API_KEY and RUN_LIVE=1 to run the live proxy + openai SDK call.")


## 5. Payoff: clean citations for any OpenAI client

| | Without the proxy | With the proxy |
|---|---|---|
| `GET /v1/models` | client hardcodes model names | real SAIA list (`fetch: true` works) |
| Assistant reply | answer **+ hundreds of lines** of `References:` dump | answer **+ compact numbered Sources** |
| Client code | bespoke SAIA + ARCANA handling | the **unmodified** OpenAI SDK |

For LibreChat: set the custom endpoint's `baseURL` to the proxy and
`models: { fetch: true }` — done.


## Notes & caveats

- **Example, not core.** A client library shouldn't ship a server. Vendor this
  code into your own service — that is exactly what the production
  `avor-adapter` does.
- **Sync vs. async / streaming.** This example uses the synchronous `SAIAClient`
  for clarity. A proxy under real streaming load should use an async stack
  (`httpx.AsyncClient`) and stream SSE through; wrapping the sync client in a
  hot streaming path blocks the event loop. The reference parser
  (`saia_python.arcana_references`) is pure and async-safe, so it works in
  either model.
- **Building blocks vs. glue.** `saia_python` gives you `list_raw()`,
  `arcana_references`, and the native client. The product-specific glue — key
  mapping, system prompts, filename→URL links, renderer quirks — stays in your
  adapter.
